In [3]:
import numpy as np
from collections import Counter

class Node:
    """Represents a single node in the Decision Tree."""
    def __init__(self, feature=None, threshold=None, left=None, right=None, *, value=None):
        self.feature = feature          # Index of feature to split on
        self.threshold = threshold      # Threshold value for split
        self.left = left                # Left child node (<= threshold)
        self.right = right              # Right child node (> threshold)
        self.value = value              # Class label if leaf node

    def is_leaf(self):
        return self.value is not None


class DecisionTreeClassifierScratch:
    def __init__(self, max_depth=10, min_samples_split=2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.root = None

    def _entropy(self, y):
        """Calculates Entropy for label array y."""
        hist = np.bincount(y)
        ps = hist / len(y)
        return -np.sum([p * np.log2(p) for p in ps if p > 0])

    def _split(self, X_column, threshold):
        """Splits feature array into left and right indices based on threshold."""
        left_idxs = np.argwhere(X_column <= threshold).flatten()
        right_idxs = np.argwhere(X_column > threshold).flatten()
        return left_idxs, right_idxs

    def _information_gain(self, X_column, y, threshold):
        """Calculates Information Gain for a potential split."""
        parent_entropy = self._entropy(y)

        # Create split
        left_idxs, right_idxs = self._split(X_column, threshold)
        if len(left_idxs) == 0 or len(right_idxs) == 0:
            return 0

        # Calculate weighted average child entropy
        n = len(y)
        n_l, n_r = len(left_idxs), len(right_idxs)
        e_l, e_r = self._entropy(y[left_idxs]), self._entropy(y[right_idxs])
        child_entropy = (n_l / n) * e_l + (n_r / n) * e_r

        # Information gain = reduction in entropy
        return parent_entropy - child_entropy

    def _best_split(self, X, y, feat_idxs):
        """Finds the best feature and threshold to split on."""
        best_gain = -1
        split_idx, split_thresh = None, None

        for feat_idx in feat_idxs:
            X_column = X[:, feat_idx]
            thresholds = np.unique(X_column)

            for threshold in thresholds:
                gain = self._information_gain(X_column, y, threshold)
                if gain > best_gain:
                    best_gain = gain
                    split_idx = feat_idx
                    split_thresh = threshold

        return split_idx, split_thresh

    def _build_tree(self, X, y, depth=0):
        n_samples, n_feats = X.shape
        n_labels = len(np.unique(y))

        # Stopping criteria: pure node, max depth reached, or too few samples
        if (depth >= self.max_depth or n_labels == 1 or n_samples < self.min_samples_split):
            most_common_label = Counter(y).most_common(1)[0][0]
            return Node(value=most_common_label)

        feat_idxs = list(range(n_feats))
        best_feat, best_thresh = self._best_split(X, y, feat_idxs)

        # If no split improves information gain
        if best_feat is None:
            most_common_label = Counter(y).most_common(1)[0][0]
            return Node(value=most_common_label)

        # Recursively build subtrees
        left_idxs, right_idxs = self._split(X[:, best_feat], best_thresh)
        left = self._build_tree(X[left_idxs, :], y[left_idxs], depth + 1)
        right = self._build_tree(X[right_idxs, :], y[right_idxs], depth + 1)

        return Node(feature=best_feat, threshold=best_thresh, left=left, right=right)

    def fit(self, X, y):
        """Fits the Decision Tree model on dataset."""
        self.root = self._build_tree(X, y)

    def _traverse_tree(self, x, node):
        """Recursively traverses the tree to predict class for a single sample."""
        if node.is_leaf():
            return node.value

        if x[node.feature] <= node.threshold:
            return self._traverse_tree(x, node.left)
        return self._traverse_tree(x, node.right)

    def predict(self, X):
        """Predicts class labels for sample array X."""
        return np.array([self._traverse_tree(x, self.root) for x in X])


# ================= Example Usage (Iris Dataset) =================
if __name__ == "__main__":
    from sklearn.datasets import load_iris
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import accuracy_score

    # Load dataset
    data = load_iris()
    X, y = data.data, data.target

    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Instantiate and fit model
    clf = DecisionTreeClassifierScratch(max_depth=4)
    clf.fit(X_train, y_train)

    # Make predictions
    predictions = clf.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)

    print(f"Iris Dataset Test Accuracy: {accuracy * 100:.2f}%")

Iris Dataset Test Accuracy: 100.00%
